# Simple linear workflow

In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict
from dotenv import load_dotenv
import os

In [2]:
load_dotenv()

True

In [3]:
model = ChatOpenAI(
    model = "openai/gpt-4o-mini",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    max_tokens = 500
)

In [4]:
class LLMState(TypedDict):

    question: str
    answer: str

In [5]:
def llm_qa(state: LLMState) -> LLMState:

    # extract the question from state
    question = state['question']
    
    # from prompt
    prompt = f'Answer the following question {question}'

    # ask that question to LLM
    answer = model.invoke(prompt).content

    # Update answer in state
    state["answer"] = answer

    return state

In [6]:
graph = StateGraph(LLMState)

In [7]:
graph.add_node("llm_qa", llm_qa)

In [8]:
graph.add_edge(START, "llm_qa")
graph.add_edge("llm_qa", END)

In [9]:
workflow = graph.compile()

In [10]:
initial_state = {'question': 'How far is moon from earth'}

In [11]:
final_state = workflow.invoke(initial_state)

In [12]:
print(final_state)

{'question': 'How far is moon from earth', 'answer': "The average distance from the Earth to the Moon is about 384,400 kilometers (about 238,855 miles). However, this distance can vary slightly due to the Moon's elliptical orbit, ranging from about 363,300 kilometers (225,623 miles) at its closest (perigee) to about 405,500 kilometers (251,966 miles) at its furthest (apogee)."}


In [13]:
model.invoke("How far is moon from earth").content

"The average distance from the Earth to the Moon is about 238,855 miles (384,400 kilometers). This distance can vary slightly due to the Moon's elliptical orbit around the Earth. At its closest point, this distance is about 225,623 miles (363,104 kilometers), and at its farthest point, it is about 252,088 miles (405,696 kilometers)."